In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0, max_retries=2)

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [6]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [7]:
intial_state = {'title': 'Rise of Modi in India'}
final_state = workflow.invoke(intial_state)
print(final_state)

{'title': 'Rise of Modi in India', 'outline': '**Blog Title (working):** *The Rise of Narendra\u202fModi: From Gujarat Chief Minister to India’s Dominant Prime Minister*  \n\n---\n\n### 1. Introduction  \n- **Hook:** A brief anecdote or statistic that captures Modi’s political impact (e.g., “In 2024, the Bharatiya Janata Party (BJP) won 46\u202f% of the popular vote, a record for a single party in post‑independence India”).  \n- **Purpose of the post:** Explain why understanding Modi’s ascent is important for readers interested in Indian politics, governance, and global affairs.  \n- **Scope & methodology:** Outline the chronological‑thematic approach and note reliance on publicly available data, academic analyses, and reputable news sources.\n\n---\n\n### 2. Early Life and Formative Influences  \n- **Family background & childhood in Vadnagar, Gujarat**  \n- **Education and early political exposure** (RSS affiliation, student activism)  \n- **Key mentors and ideological grounding** (e.